In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from combra import data, angles
from combra.metrics import (
    metrics_vs_n,
    convergence_stats, print_convergence_report,
    summarize_metric_distribution,
    plot_wdist_convergence_grid, plot_metric_distribution,
)
from combra.metrics.compare import find_ethalon

# Angle binning step (degrees). Used by generation AND the W-dist computation.
# generate_angles overwrites parquets, so changing STEP and re-running generation
# drops any previously-stored step. Files already containing STEP are skipped.
STEP = 2.0
ANGLES_ROOT = Path('./data/angles')
MIN_SEG_LEN = 5.0

def sweep_dir(h5_path):
    return angles.angles_out_dir(ANGLES_ROOT, h5_path, MIN_SEG_LEN)


In [ ]:
# Per-resolution sources for the 3x3 convergence grid. Each row = one resolution;
# the 'real' H5 doubles as ethalon (largest-N parquet) and as source of the
# original self-convergence curve.
SOURCES = {
    256: {
        'real':   './data/h5/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360.h5',
        'san':    './data/h5/gen_san_256x256_N100_000.h5',
        'diffit': './data/h5/00017-diffit-256-gpus2-batch192_N10000.h5',
    },
    512: {
        'real':   './data/h5/o_bc_left_4x_1536_1024x1024_512x512_rgb_N360.h5',
        'san':    './data/h5/gen_san_512x512_N100_000.h5',
        'diffit': './data/h5/00018-diffit-512-gpus4-batch256_N10000.h5',
    },
    1024: {
        'real':   './data/h5/o_bc_left_4x_1536_1024x1024_1024x1024_rgb_N360.h5',
    },
}

# N sweep per (kind, resolution). Bounded by per-class dataset size of each H5 —
# diffit at 512 only has 1k/class so its sweep stops there. The largest N in each
# list also serves as the ethalon for that (resolution, kind).
N_VALUES = {
    'real':   {256:  [100, 150, 200, 250, 300, 360],
               512:  [100, 150, 200, 250, 300, 360],
               1024: [100, 150, 200, 250, 300, 360]},
    'san':    {256:  [100, 250, 360, 500, 1000, 2500, 5000, 10000],
               512:  [100, 250, 360, 500, 1000, 2500, 5000, 10000]},
    'diffit': {256:  [100, 250, 360, 500, 1000, 2500, 5000, 10000],
               512:  [100, 250, 360, 500, 1000, 2500, 5000, 10000]},
}

# Diffit and san indexed classes in different orders -> per-generator gen->real map.
CLASS_MAP_PER_KIND = {
    'san':    {'class_0': 'class_Ultra_Co25', 'class_1': 'class_Ultra_Co11', 'class_2': 'class_Ultra_Co6_2'},
    'diffit': {'class_0': 'class_Ultra_Co11', 'class_1': 'class_Ultra_Co25', 'class_2': 'class_Ultra_Co6_2'},
}
types_dict = {
    "Ultra_Co11": "мелкие зерна",
    "Ultra_Co25": "средние зерна",
    "Ultra_Co8": "средне-мелкие зерна",
    "Ultra_Co6_2": "крупные зерна",
    "Ultra_Co15": "средне-мелкие зерна",
}
CLASSES = ['class_Ultra_Co11', 'class_Ultra_Co25', 'class_Ultra_Co6_2']
GRAIN_LABEL = {'class_Ultra_Co11': 'small grain',
               'class_Ultra_Co25': 'medium grain',
               'class_Ultra_Co6_2': 'large grain'}
LABELS = {'real': 'original', 'san': 'san', 'diffit': 'diffit '}

# Stats config (used in the convergence + gain-distribution sections).
# Angle-distribution metrics from the parquet sweep: W-dist family + Gaussian-fit
# relative errors. Image-feature FID/CMMD/FD-DINOv2 are intentionally not computed.
METRICS = ['w1', 'w2', 'circular_w1', 'circular_w2',
           'mu1', 'mu2', 'sigma1', 'sigma2', 'amp1', 'amp2']
KINDS = ['san', 'diffit']
# Main "plateau-regime" endpoints used for rel_err_abs_% and the T3 main column.
ENDPOINTS_BY_KIND = {'real': (100, 300), 'san': (1000, 10000), 'diffit': (1000, 10000)}
# Pre-plateau endpoints — earlier-N pair shown alongside the main one in T3.
PRE_ENDPOINTS_BY_KIND = {'san': (360, 1000), 'diffit': (360, 1000)}
# Display names for the per-resolution report tables and plot annotations.
KIND_DISPLAY = {'real': 'Real', 'san': 'SAN', 'diffit': 'DiffiT'}

# Generation

In [ ]:
GEN_KW = dict(workers=20, angles_tol=3, min_segment_len=5.0,
              keep_contours=False, chunksize=64)

for resolution, group in SOURCES.items():
    print(f'\n=== resolution {resolution}x{resolution} ===')
    for kind, h5_path in group.items():
        data.generate_angles_sweep(
            h5_path, sweep_dir(h5_path),
            ns=N_VALUES[kind][resolution], step=STEP,
            types_dict=types_dict, tag=kind,
            run_meta={'family': kind, 'resolution': resolution, 'tags': [], 'notes': ''},
            **GEN_KW,
        )

# Convergence grid

In [ ]:
def collect_records():
    """Walk SOURCES once and return (records_by_panel, df_metrics).

    Per panel the ethalon is the resolution's largest-N real parquet, so every
    san/diffit curve is (N sampled) vs (real ethalon). The 'real' kind is the
    self-convergence baseline (N real vs real ethalon). metrics_vs_n dedupes any
    stray same-N parquet, so the real curve converges cleanly to 0.
    """
    records_by_panel, rows = {}, []
    for resolution, group in SOURCES.items():
        ethalon = find_ethalon(sweep_dir(group['real']))
        if ethalon is None:
            print(f'[row {resolution}] no ethalon parquet — skip row')
            continue
        print(f'row {resolution}: ethalon = {ethalon}')
        for kind, h5 in group.items():
            d = sweep_dir(h5)
            if not d.exists():
                continue
            recs = metrics_vs_n(d, str(ethalon),
                                class_map=CLASS_MAP_PER_KIND.get(kind, {}),
                                step=STEP,
                                allowed_ns=set(N_VALUES[kind][resolution]))
            records_by_panel[(resolution, kind)] = recs
            rows.extend({'kind': kind, 'resolution': resolution, **r} for r in recs)
    return records_by_panel, pd.DataFrame.from_records(rows)

records_by_panel, df_metrics = collect_records()

result = convergence_stats(df_metrics, METRICS, ENDPOINTS_BY_KIND,
                           expected_points={'real': 6, 'san': 8, 'diffit': 8},
                           pre_endpoints_by_kind=PRE_ENDPOINTS_BY_KIND)

# Per-subplot annotation: per-kind Kendall p, plateau-fit a/b for
# |m|(N) = a + b/sqrt(N), and its R^2. The plot fn colors each kind's label
# to match its curve, so the box doubles as a per-panel legend.
_STAT_COLS = ['kendall_p', 'a_hat', 'b_hat', 'fit_r2']

def _panel_stats(sub):
    return {row['kind']: {col: row[col] for col in _STAT_COLS}
            for _, row in sub.iterrows()}

# y-axis label per metric. Data points are drawn as a bare |metric| scatter on a
# log y-axis; the only line per panel is the a + b/sqrt(N) plateau fit (dotted).
METRIC_LABEL = {
    'w1': 'W₁-dist', 'w2': 'W₂-dist',
    'circular_w1': 'circ. W₁', 'circular_w2': 'circ. W₂',
    'mu1':    'μ₁ rel. err.', 'mu2':    'μ₂ rel. err.',
    'sigma1': 'σ₁ rel. err.', 'sigma2': 'σ₂ rel. err.',
    'amp1':   'A₁ rel. err.', 'amp2':   'A₂ rel. err.',
}

for metric in METRICS:
    sub_metric = result[result['metric'] == metric]
    panel_ann = {(int(res), cls): _panel_stats(g)
                 for (res, cls), g in sub_metric.groupby(['resolution', 'class'])}
    save_path = f'{metric}_convergence_step{STEP}.png'
    fig = plot_wdist_convergence_grid(
        records_by_panel, classes=CLASSES,
        kind_labels=LABELS, grain_labels=GRAIN_LABEL,
        row_keys=list(SOURCES), save_path=save_path,
        metric=metric, y_label=f'|{METRIC_LABEL[metric]}|',
        abs_values=True, log_y=True, fit_line=True, connect_points=False,
        panel_annotations=panel_ann,
        annot_kind_labels=KIND_DISPLAY,
    )
    print(f'saved: {save_path}')
    fig.show()

print_convergence_report(result, METRICS, KINDS, ENDPOINTS_BY_KIND,
                         step=STEP, kind_display=KIND_DISPLAY,
                         pre_endpoints_by_kind=PRE_ENDPOINTS_BY_KIND)

# All-metrics panel

In [ ]:
from combra.metrics import plot_metrics_grid

# Fix (class, resolution) and tile all seven metrics in one figure — the
# transpose of the per-metric convergence grids above (same records_by_panel).
# Each subplot is one metric vs N with the original/san/diffit curves overlaid,
# so a single panel shows how every distribution metric converges for that class.
GRID_RES = 256
for cls in CLASSES:
    fig = plot_metrics_grid(
        records_by_panel, GRID_RES, cls, METRICS,
        metric_labels=METRIC_LABEL, kind_labels=LABELS,
        title=f'{GRAIN_LABEL[cls]} ({cls}) — {GRID_RES}×{GRID_RES}',
        save_path=f'metrics_grid_{cls}_{GRID_RES}_step{STEP}.png',
    )
    fig.show()

# Gain distribution (experimental)

In [ ]:
from combra.metrics import plot_distribution_grid

STEP_PCT   = 1.5      # bin width for gain_% subplots (percentage points)
STEP_ALPHA = 0.1      # bin width for alpha subplots (dimensionless)
Y_LIM         = (0.0, 0.5)
GAIN_X_LIM    = (-20, 60)
ALPHA_X_LIM   = (-1.0, 2.0)

RESOLUTIONS = [256, 512]
# Column names match T3 of the convergence report (gain_%_a->b / alpha_a->b).
GAIN_COLS  = [('pre_rel_err_abs_%', 'gain_%_360->10^3'),
              ('rel_err_abs_%',     'gain_%_10^3->10^4')]
ALPHA_COLS = [('pre_alpha', 'alpha_360->10^3'),
              ('alpha',     'alpha_10^3->10^4')]
KIND_COLOR = {'san':    'rgba(31,119,180,0.95)',
              'diffit': 'rgba(255,127,14,0.95)'}

for col, label in GAIN_COLS + ALPHA_COLS:
    print(f'\n  [{label}]')
    for tag, line in summarize_metric_distribution(result, col, KINDS, RESOLUTIONS).items():
        print(f'    {tag}:  {line}')

# Shared 2x2 grid config: rows=RESOLUTIONS, cols=metric columns, overlay by kind.
_GRID_KW = dict(kind_color=KIND_COLOR, kind_display=KIND_DISPLAY,
                y_lim=Y_LIM, row_label_fn=lambda r: f'{r}x{r}')

# gain refs: 0 = no sampling-only gain left.
save_gain = f'gain_dist_step{STEP}_binstep{STEP_PCT}.png'
fig_gain = plot_distribution_grid(result, RESOLUTIONS, GAIN_COLS, KINDS,
                                  bin_step=STEP_PCT, x_lim=GAIN_X_LIM,
                                  ref_lines=[(0.0, 'dash')], save_path=save_gain, **_GRID_KW)
print(f'saved: {save_gain}')
fig_gain.show()

# alpha refs: 0 = no improvement, 0.5 = ideal Monte-Carlo decay.
save_alpha = f'alpha_dist_step{STEP}_binstep{STEP_ALPHA}.png'
fig_alpha = plot_distribution_grid(result, RESOLUTIONS, ALPHA_COLS, KINDS,
                                   bin_step=STEP_ALPHA, x_lim=ALPHA_X_LIM,
                                   ref_lines=[(0.0, 'dash'), (0.5, 'dot')],
                                   save_path=save_alpha, **_GRID_KW)
print(f'saved: {save_alpha}')
fig_alpha.show()